In [2]:
import pandas as pd
import numpy as np
import os

processed_dir = "../data/processed/"
print("Loading clean artifacts...")

# Load the data we prepped in Notebook 1
metadata = pd.read_csv(os.path.join(processed_dir, 'master_metadata.csv'))
ratings = pd.read_csv(os.path.join(processed_dir, 'ratings_sample.csv'))

# Ensure there are no NaNs in our text column before vectorizing
metadata['combined_features'] = metadata['combined_features'].fillna('')

# Create a reverse mapping of movie titles to their indices for fast lookup
indices = pd.Series(metadata.index, index=metadata['original_title']).drop_duplicates()

print(f"Loaded {len(metadata)} movies and {len(ratings)} ratings.")

Loading clean artifacts...
Loaded 43794 movies and 500000 ratings.


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("Vectorizing text features...")

# Initialize the vectorizer, removing common English words (stop_words)
tfidf = TfidfVectorizer(stop_words='english')

# Create the TF-IDF sparse matrix (Takes virtually no RAM)
tfidf_matrix = tfidf.fit_transform(metadata['combined_features'])

print(f"TF-IDF Matrix created! Shape: {tfidf_matrix.shape}")

Vectorizing text features...
TF-IDF Matrix created! Shape: (43794, 72868)


In [4]:
from surprise import Reader, Dataset, SVD

print("Training Collaborative Filter (SVD)...")

# Surprise requires a specific 'Reader' format to understand the rating scale
reader = Reader(rating_scale=(0.5, 5.0))

# Load the Pandas dataframe into a Surprise Dataset
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# Initialize the SVD algorithm
svd_model = SVD(n_epochs=20, lr_all=0.005, reg_all=0.02)

# Build the training set and fit the model (This might take 10-20 seconds)
trainset = data.build_full_trainset()
svd_model.fit(trainset)

print("SVD Model trained successfully!")

Training Collaborative Filter (SVD)...
SVD Model trained successfully!


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def hybrid_recommendation(user_id, title, top_n=10):
    if title not in indices:
        return f"Error: '{title}' not found in database."
        
    # 1. Get the index of the target movie
    idx = indices[title]
    
    # 2. Compute similarity for ONLY the target movie against all others (Memory Safe!)
    target_vector = tfidf_matrix[idx]
    sim_scores = cosine_similarity(target_vector, tfidf_matrix).flatten()
    
    # 3. Get the indices of the top 50 most similar movies
    # np.argsort sorts ascending, so we take the last 51 and reverse it, skipping the first one (itself)
    top_50_indices = sim_scores.argsort()[-51:-1][::-1]
    
    # 4. Extract those 50 movies
    sim_movies = metadata.iloc[top_50_indices].copy()
    
    # 5. Predict what this specific user would rate these 50 similar movies
    sim_movies['predicted_rating'] = sim_movies['movieId'].apply(lambda x: svd_model.predict(user_id, x).est)
    
    # 6. Sort the 50 movies by the SVD predicted rating
    sim_movies = sim_movies.sort_values('predicted_rating', ascending=False)
    
    return sim_movies[['original_title', 'predicted_rating']].head(top_n)

print("Hybrid Engine ready! Try running: hybrid_recommendation(user_id=1, title='Toy Story')")

Hybrid Engine ready! Try running: hybrid_recommendation(user_id=1, title='Toy Story')


In [6]:
import pickle
import os

print("Exporting trained models for deployment...")

models_dir = "../artifacts/saved_models/"
os.makedirs(models_dir, exist_ok=True)

# Save the trained SVD model
with open(os.path.join(models_dir, 'svd_model.pkl'), 'wb') as f:
    pickle.dump(svd_model, f)

# Save the TF-IDF Matrix and the Vectorizer
with open(os.path.join(models_dir, 'tfidf_matrix.pkl'), 'wb') as f:
    pickle.dump(tfidf_matrix, f)

with open(os.path.join(models_dir, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(tfidf, f)

# Save the indices mapping
with open(os.path.join(models_dir, 'movie_indices.pkl'), 'wb') as f:
    pickle.dump(indices, f)

print("Models successfully saved to /artifacts/saved_models/")

Exporting trained models for deployment...
Models successfully saved to /artifacts/saved_models/
